In [16]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
import io


# Input data
data = {
    'Today': ['14-Feb-26'] * 12,
    'Maturity Date': [
        '12-Aug-28', '12-Feb-29', '12-Aug-31', '12-Aug-32', '12-Aug-33',
        '12-Aug-34', '12-Aug-35', '12-Aug-37', '12-Aug-39', '12-Aug-40',
        '12-Feb-42', '12-Feb-55'
    ],
    'Yield': [4.00, 4.05, 4.39, 4.58, 4.98, 5.43, 5.83, 6.30, 6.53, 6.53, 6.15, 6.15],
    'clean price': [105.505, 105.234, 112.426, 108.78, 114.31, 99.768, 107.515, 
                    104.913, 109.789, 89.131, 107.095, 107.633],
    'duration': [2.39, 2.84, 4.778, 5.58, 6.11, 7.02, 7.33, 8.33, 9.03, 10.13, 10.33, 13.86],
    'moddur': [2.30, 2.73, 4.58, 5.34, 5.82, 6.66, 6.93, 7.84, 8.48, 9.51, 9.73, 13.06],
    'coupon': [6.35, 5.94, 6.95, 6.15, 7.30, 5.40, 6.85, 6.90, 7.60, 5.35, 6.85, 6.714]
}

df = pd.DataFrame(data)

# Convert dates to datetime
df['Today'] = pd.to_datetime(df['Today'], format='%d-%b-%y')
df['Maturity Date'] = pd.to_datetime(df['Maturity Date'], format='%d-%b-%y')

# Calculate time to maturity in years
df['TTM_years'] = (df['Maturity Date'] - df['Today']).dt.days / 365
# Calculate 90-day forward date and time to maturity
forward_date = df['Today'].iloc[0] + timedelta(days=90)
df['TTM_forward_years'] = (df['Maturity Date'] - forward_date).dt.days / 365

print(f"Today's date: {df['Today'].iloc[0].strftime('%d-%b-%Y')}")
print(f"Forward date (90d): {forward_date.strftime('%d-%b-%Y')}\n")

# Build yield curve interpolation (current curve)
# Remove any duplicate TTM values
df_curve = df[['TTM_years', 'Yield']].drop_duplicates(subset='TTM_years').sort_values('TTM_years')

# Create interpolation function (linear, can change to cubic)
yield_curve = interp1d(
    df_curve['TTM_years'], 
    df_curve['Yield'], 
    kind='linear',
    fill_value='extrapolate'
)

# Extract year and sort by maturity
df['Maturity Year'] = df['Maturity Date'].dt.year
df = df.sort_values('Maturity Year').reset_index(drop=True)

# Calculate 90-day roll (yield change from rolling down the curve)
df['Forward_Yield'] = yield_curve(df['TTM_forward_years'])
df['90d rolldown (yield bps)'] = (df['Forward_Yield'] - df['Yield']) * 100

# Calculate 90-day carry in yield bps
# Step 1: Calculate carry as price return (running yield over 90 days)
carry_price_return = (df['coupon'] / df['clean price']) * (90/365)*-1

# Step 2: Convert price return to yield bps using modified duration
# Carry (yield bps) = Carry (price %) / Modified Duration * 100
df['90d carry (yield bps)'] = (carry_price_return / df['moddur']) * 10000

# Total carry + roll
df['Total C+R'] = df['90d carry (yield bps)'] + df['90d rolldown (yield bps)']

Today's date: 14-Feb-2026
Forward date (90d): 15-May-2026



In [14]:
# Create output table with requested columns
output = df[[
    'Maturity Date', 
    'Yield', 
    'moddur', 
    '90d rolldown (yield bps)', 
    '90d carry (yield bps)', 
    'Total C+R'
]].copy()

# Format maturity date
output['Maturity Date'] = output['Maturity Date'].dt.strftime('%d-%b-%y')

# Round numerical columns
output['Yield'] = output['Yield'].round(2)
output['moddur'] = output['moddur'].round(2)
output['90d rolldown (yield bps)'] = output['90d rolldown (yield bps)'].round(1)
output['90d carry (yield bps)'] = output['90d carry (yield bps)'].round(1)
output['Total C+R'] = output['Total C+R'].round(1)

# Display results
print("=" * 100)
print("90-Day Carry and Roll Analysis (in Yield bps)")
print("=" * 100)
print(output.to_string(index=False))

# Optional: Export to CSV
output.to_csv('carry_and_roll_analysis.csv', index=False)
print("\n✓ Results exported to 'carry_and_roll_analysis.csv'")

import os
csv_path = os.path.abspath('carry_and_roll_analysis.csv')
print(f"\n✓ Results exported to: {csv_path}")

# Show summary statistics
print("\n" + "=" * 100)
print("Summary Statistics")
print("=" * 100)
print(f"Average 90d Carry: {output['90d carry (yield bps)'].mean():.1f} bps")
print(f"Average 90d Roll: {output['90d rolldown (yield bps)'].mean():.1f} bps")
print(f"Average Total C+R: {output['Total C+R'].mean():.1f} bps")
print(f"Best Total C+R: {output['Total C+R'].max():.1f} bps (Maturity: {output.loc[output['Total C+R'].idxmax(), 'Maturity Date']})")
print(f"Worst Total C+R: {output['Total C+R'].min():.1f} bps (Maturity: {output.loc[output['Total C+R'].idxmin(), 'Maturity Date']})")


90-Day Carry and Roll Analysis (in Yield bps)
Maturity Date  Yield  moddur  90d rolldown (yield bps)  90d carry (yield bps)  Total C+R
    12-Aug-28   4.00    2.30                      -2.4                  -64.5      -67.0
    12-Feb-29   4.05    2.73                      -2.4                  -51.0      -53.4
    12-Aug-31   4.39    4.58                      -3.4                  -33.3      -36.6
    12-Aug-32   4.58    5.34                      -4.7                  -26.1      -30.8
    12-Aug-33   4.98    5.82                      -9.9                  -27.1      -36.9
    12-Aug-34   5.43    6.66                     -11.1                  -20.0      -31.1
    12-Aug-35   5.83    6.93                      -9.9                  -22.7      -32.5
    12-Aug-37   6.30    7.84                      -5.8                  -20.7      -26.5
    12-Aug-39   6.53    8.48                      -2.8                  -20.1      -23.0
    12-Aug-40   6.53    9.51                       0.0          

In [17]:
# Create PDF
pdf_filename = 'carry_and_roll_analysis.pdf'

# Generate matplotlib charts first
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

maturity_years = df['Maturity Year'].values
carry = df['90d carry (yield bps)'].values
roll = df['90d rolldown (yield bps)'].values
total = df['Total C+R'].values

x_pos = np.arange(len(maturity_years))
bar_width = 0.6

# Chart 1: Carry + Roll
bars1 = axes[0].bar(x_pos, carry, bar_width, label='Carry', color='#5B9BD5')
bars2 = axes[0].bar(x_pos, roll, bar_width, bottom=carry, label='Roll Down', color='#70AD47')
axes[0].set_title('Carry + Roll', fontsize=9, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(maturity_years, rotation=45, fontsize=7)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].grid(True, axis='y', alpha=0.3, linestyle='--')
axes[0].set_axisbelow(True)
axes[0].legend(fontsize=7, loc='upper left')

# Add labels for Chart 1 (stacked)
for i, (c, r, t) in enumerate(zip(carry, roll, total)):
    # Label for total at the outer edge
    y_pos = t
    va = 'bottom' if t >= 0 else 'top'
    offset = 2 if t >= 0 else -2
    axes[0].text(i, y_pos + offset, f'{int(round(t))}', ha='center', va=va, fontsize=6, fontweight='bold')

# Chart 2: Carry only
bars3 = axes[1].bar(x_pos, carry, bar_width, color='#5B9BD5')
axes[1].set_title('Carry', fontsize=9, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(maturity_years, rotation=45, fontsize=7)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].grid(True, axis='y', alpha=0.3, linestyle='--')
axes[1].set_axisbelow(True)

# Add labels for Chart 2
for i, c in enumerate(carry):
    y_pos = c
    va = 'bottom' if c >= 0 else 'top'
    offset = 2 if c >= 0 else -2
    axes[1].text(i, y_pos + offset, f'{int(round(c))}', ha='center', va=va, fontsize=6, fontweight='bold')

# Chart 3: Roll only
bars4 = axes[2].bar(x_pos, roll, bar_width, color='#70AD47')
axes[2].set_title('Roll Down', fontsize=9, fontweight='bold')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(maturity_years, rotation=45, fontsize=7)
axes[2].axhline(y=0, color='black', linewidth=0.5)
axes[2].grid(True, axis='y', alpha=0.3, linestyle='--')
axes[2].set_axisbelow(True)

# Add labels for Chart 3
for i, r in enumerate(roll):
    y_pos = r
    va = 'bottom' if r >= 0 else 'top'
    offset = 2 if r >= 0 else -2
    axes[2].text(i, y_pos + offset, f'{int(round(r))}', ha='center', va=va, fontsize=6, fontweight='bold')

# Set equal y-axis limits for all charts with extra padding for labels
y_min = min(carry.min(), roll.min(), total.min()) * 1.2
y_max = max(carry.max(), roll.max(), total.max()) * 1.2
for ax in axes:
    ax.set_ylim([y_min, y_max])
    ax.tick_params(labelsize=7)

plt.tight_layout()

# Save charts to buffer
chart_buffer = io.BytesIO()
plt.savefig(chart_buffer, format='png', dpi=150, bbox_inches='tight')
chart_buffer.seek(0)
plt.close()

# Create PDF with ReportLab
doc = SimpleDocTemplate(pdf_filename, pagesize=A4, topMargin=0.5*inch, bottomMargin=0.5*inch)
story = []

# Add table
table_data = [['Maturity\nDate', 'Yield', 'Mod\nDur', '90d Rolldown\n(yield bps)', 
               '90d Carry\n(yield bps)', 'Total\nC+R']]
for _, row in output.iterrows():
    table_data.append([
        row['Maturity Date'],
        f"{row['Yield']:.2f}",
        f"{row['moddur']:.2f}",
        f"{row['90d rolldown (yield bps)']:.1f}",
        f"{row['90d carry (yield bps)']:.1f}",
        f"{row['Total C+R']:.1f}"
    ])

table = Table(table_data, colWidths=[0.9*inch, 0.6*inch, 0.6*inch, 1.0*inch, 1.0*inch, 0.7*inch])
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#4472C4')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 8),
    ('FONTSIZE', (0, 1), (-1, -1), 7),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#E7E6E6')]),
]))

story.append(table)
story.append(Spacer(1, 0.2*inch))

# Add charts
chart_img = Image(chart_buffer, width=7*inch, height=2.1*inch)
story.append(chart_img)

# Build PDF
doc.build(story)

print(f"✓ PDF created: {pdf_filename}")

✓ PDF created: carry_and_roll_analysis.pdf
